In [ ]:
# Import functions from preprocessing.py
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.gaussian_process.kernels import Matern


# Add the path to the src directory
sys.path.append(os.path.abspath(os.path.join('..', 'data')))
from generation import generate_data


# Data generation
for seed_generate in range(1, 101):
    for n_i in [4, 6, 8, 10, 20]:
        B = 100  # Number of regions
        sigmasq_true = 5
        phi_true = 4
        tausq_true = 0.25
        nu_true=0.5
        beta_true = 8
        # Generate data
        y, x, w, e, s, region_assignments = generate_data(B, n_i, sigmasq=sigmasq_true, length_scale=1/phi_true, nu=nu_true, seed=seed_generate, beta_true=beta_true, tausq_true=tausq_true)
        
        # Jumbled data generation
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        input_dim = 1

        y = torch.tensor(y, dtype=torch.float32, device=device)
        x = torch.tensor(x, dtype=torch.float32, device=device)
        w = torch.tensor(w, dtype=torch.float32, device=device)
        e = torch.tensor(e, dtype=torch.float32, device=device)
        s = torch.tensor(s, dtype=torch.float32, device=device)
        region_assignments = torch.tensor(region_assignments, dtype=torch.int64, device=device)  # Region indices as integers
        unique_regions = torch.unique(region_assignments)

        # Jumble x and s within regions
        x_jumbled_within_regions = torch.zeros_like(x)
        s_jumbled_within_regions = torch.zeros_like(s)
        torch.manual_seed(5556)  # Set a seed for reproducibility
        perm_x = torch.randperm(n_i)
        perm_s = torch.randperm(n_i)

        for i, region in enumerate(unique_regions):
            # Get indices for the current region
            indices = torch.where(region_assignments == region)[0]
            
            # Assign the shuffled values back
            x_jumbled_within_regions[indices] = x[indices[perm_x]]
            s_jumbled_within_regions[indices] = s[indices[perm_s]]

        # Convert perm_s into a permutation matrix
        perm_matrix_s = torch.zeros(n_i, n_i, dtype=torch.float32, device=device)
        perm_matrix_s[torch.arange(n_i), perm_s] = 1

        # Convert perm_x into a permutation matrix
        perm_matrix_x = torch.zeros(n_i, n_i, dtype=torch.float32, device=device)
        perm_matrix_x[torch.arange(n_i), perm_x] = 1

        # Create the directory structure if it doesn't exist
        output_dir = os.path.join('..', 'data', f"B_{B}_n_{n_i}")
        os.makedirs(output_dir, exist_ok=True)

        # Save the tensors to a file
        output_file = os.path.join(output_dir, f"data_seed_{seed_generate}.pt")
        torch.save({
            'y': y,
            'region_assignments': region_assignments,
            'x': x,
            'w': w,
            'e': e,
            's': s,
            'x_jumbled_within_regions': x_jumbled_within_regions,
            's_jumbled_within_regions': s_jumbled_within_regions,
            'perm_matrix_x': perm_matrix_x,
            'perm_matrix_s': perm_matrix_s,
            'sigmasq_true': sigmasq_true,
            'phi_true': phi_true,
            'beta_true': beta_true,
            'nu_true': nu_true,
            'tausq_true': tausq_true
        }, output_file)

        print(f"Data saved to {output_file}")


Data saved to ../data/B_100_n_4_seed_1/data.pt
Data saved to ../data/B_100_n_6_seed_1/data.pt
Data saved to ../data/B_100_n_8_seed_1/data.pt
Data saved to ../data/B_100_n_10_seed_1/data.pt
Data saved to ../data/B_100_n_20_seed_1/data.pt
Data saved to ../data/B_100_n_4_seed_2/data.pt
Data saved to ../data/B_100_n_6_seed_2/data.pt
Data saved to ../data/B_100_n_8_seed_2/data.pt
Data saved to ../data/B_100_n_10_seed_2/data.pt
Data saved to ../data/B_100_n_20_seed_2/data.pt
Data saved to ../data/B_100_n_4_seed_3/data.pt
Data saved to ../data/B_100_n_6_seed_3/data.pt
Data saved to ../data/B_100_n_8_seed_3/data.pt
Data saved to ../data/B_100_n_10_seed_3/data.pt
Data saved to ../data/B_100_n_20_seed_3/data.pt
Data saved to ../data/B_100_n_4_seed_4/data.pt
Data saved to ../data/B_100_n_6_seed_4/data.pt
Data saved to ../data/B_100_n_8_seed_4/data.pt
Data saved to ../data/B_100_n_10_seed_4/data.pt


KeyboardInterrupt: 